# Multilayer perceptron tutorial

We train `mlpackage.supervised_learning.MultilayerPerceptron` (softmax output, full-batch gradient descent) on **Iris**: 4 features, 3 species. Inputs are **standardized** with `StandardScaler` fit on the training set only.

## Prerequisites and goals

**You will**
- Build a small MLP: specify `layer_sizes`, hidden `activation`, and `fit(..., learning_rate, epochs)`.
- Evaluate **accuracy** and inspect **softmax** probabilities (the class has no `score` method—use explicit comparisons).
- Visualize **2D class regions** with a **separate 2-input** network (like the logistic boundary figure).
- Compare a few **hidden layer widths** on the same train/test split.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from mlpackage.supervised_learning import MultilayerPerceptron

## Step 1: Load Iris

Three classes (setosa, versicolor, virginica), four measurements per flower. Integer labels `0,1,2` match the MLP’s **softmax** head.

In [ ]:
iris = load_iris()
X = np.asarray(iris.data, dtype=float)
y = np.asarray(iris.target, dtype=int)

feature_names = list(iris.feature_names)
target_names = list(iris.target_names)

print("X shape:", X.shape, "  y shape:", y.shape)
print("Classes:", {i: target_names[i] for i in range(3)})
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

## Step 2: Stratified train/test split

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])

## Step 3: Standardize (fit on train only)

Neural nets are sensitive to feature scale; use **train** statistics to transform both splits.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Step 4: Fit MLP on all four features

Architecture **`[4, 16, 3]`**: one hidden layer of 16 units, $\tanh$ activation, L2 regularization. Tune **`epochs`** / **`learning_rate`** if accuracy is unstable (this is full-batch GD, not mini-batches).

### Forward pass and softmax

Let `layer_sizes` be $[d_0, d_1, \ldots, d_L]$ with input $\mathbf{a}^{(0)}=\mathbf{x}$. For hidden layers $\ell=1,\ldots,L-1$,

$$
\mathbf{z}^{(\ell)} = \mathbf{W}^{(\ell)} \mathbf{a}^{(\ell-1)} + \mathbf{b}^{(\ell)}, \qquad
\mathbf{a}^{(\ell)} = \phi\big(\mathbf{z}^{(\ell)}\big),
$$

with elementwise $\phi \in \{\tanh,\sigma,\mathrm{ReLU}\}$. Output logits $\mathbf{z}^{(L)}$ feed softmax probabilities

$$
P(y=k \mid \mathbf{x}) = \frac{e^{z^{(L)}_k}}{\sum_j e^{z^{(L)}_j}}.
$$

### Loss and weight decay

For one-hot targets matching integer labels $y_i$, average **cross-entropy** is $\mathcal{L}_{\mathrm{CE}} = -\frac{1}{n}\sum_i \log p_{i,y_i}$. The implementation adds **L2** penalty $\frac{\lambda}{2}\sum_{\ell}\|\mathbf{W}^{(\ell)}\|_F^2$ with coefficient **`l2_penalty`** $=\lambda$.

**Backpropagation** applies the chain rule: for softmax output, $\delta^{(L)} = \mathbf{p}-\mathbf{y}_{\text{one-hot}}$; upstream,

$$
\delta^{(\ell)} = \big((\mathbf{W}^{(\ell+1)})^\top \delta^{(\ell+1)}\big) \odot \phi'(\mathbf{z}^{(\ell)}),
$$

with weight gradients from activations and $\delta$ signals (averaged over the batch in this code).



In [ ]:
LEARNING_RATE = 0.5
EPOCHS = 5000
RNG = 42

mlp = MultilayerPerceptron(
    layer_sizes=[4, 16, 3],
    activation="tanh",
    l2_penalty=0.01,
    rng_seed=RNG,
)
mlp.fit(X_train_s, y_train, learning_rate=LEARNING_RATE, epochs=EPOCHS)
print("Training finished.")

## Step 5: Predicted classes and softmax probabilities (test set)

Rows of **`predict_probability`** sum to 1; **`predict`** is the **argmax** class index.

In [ ]:
P_test = mlp.predict_probability(X_test_s)
y_pred = mlp.predict(X_test_s)

n_show = 10
cols = [f"P({target_names[k]})" for k in range(3)]
prob_df = pd.DataFrame(P_test[:n_show], columns=cols).round(4)
prob_df["y_true"] = [target_names[t] for t in y_test[:n_show]]
prob_df["y_pred"] = [target_names[t] for t in y_pred[:n_show]]
display(prob_df)

## Step 6: Accuracy (train & test) and per-class test hits

There is no `score` method—use direct comparison of integer labels.

In [ ]:
def accuracy(y_true: np.ndarray, y_hat: np.ndarray) -> float:
    return float(np.mean(y_true == y_hat))


y_hat_train = mlp.predict(X_train_s)
print(f"Train accuracy: {accuracy(y_train, y_hat_train):.4f}")
print(f"Test accuracy : {accuracy(y_test, y_pred):.4f}")

for c in range(3):
    mask = y_test == c
    hits = int(np.sum((y_pred == y_test) & mask))
    tot = int(np.sum(mask))
    print(f"Test {target_names[c]}: {hits}/{tot} correct")

## Step 7: 2D decision regions + test points (visualization network)

The model above maps \(\mathbb{R}^4 \to \mathbb{R}^3\). To plot regions in the plane we **refit** an MLP with **`layer_sizes=[2, 16, 3]`** on **sepal length** and **sepal width** only (columns 0 and 1), then label a fine grid with **`predict`**. This is a **teaching** picture, not the same function as the 4D model in Step 4.

In [ ]:
from pathlib import Path

i, j = 0, 1
X_train_2d = X_train_s[:, [i, j]]
X_test_2d = X_test_s[:, [i, j]]

viz = MultilayerPerceptron(
    layer_sizes=[2, 16, 3],
    activation="tanh",
    l2_penalty=0.01,
    rng_seed=RNG + 7,
)
viz.fit(X_train_2d, y_train, learning_rate=LEARNING_RATE, epochs=8000)

h = 0.02
pad = 0.4
x_min = float(min(X_train_2d[:, 0].min(), X_test_2d[:, 0].min()) - pad)
x_max = float(max(X_train_2d[:, 0].max(), X_test_2d[:, 0].max()) + pad)
y_min = float(min(X_train_2d[:, 1].min(), X_test_2d[:, 1].min()) - pad)
y_max = float(max(X_train_2d[:, 1].max(), X_test_2d[:, 1].max()) + pad)

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h),
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = viz.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(
    xx,
    yy,
    Z,
    levels=[-0.5, 0.5, 1.5, 2.5],
    colors=["#ffcccc", "#ccffcc", "#cce5ff"],
    alpha=0.85,
)

markers = ["o", "s", "^"]
edge_colors = ["black", "black", "black"]
for c in range(3):
    m = y_test == c
    plt.scatter(
        X_test_2d[m, 0],
        X_test_2d[m, 1],
        c=["crimson", "darkgreen", "navy"][c],
        marker=markers[c],
        s=70,
        edgecolors=edge_colors[c],
        linewidths=0.8,
        label=f"{target_names[c]} (test)",
        zorder=5,
    )

plt.xlabel(f"Feature 1 — {feature_names[i]} [standardized]")
plt.ylabel(f"Feature 2 — {feature_names[j]} [standardized]")
plt.title("MLP decision regions (2-feature model, test samples overlaid)")
plt.legend(loc="upper left", fontsize=9)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.tight_layout()

_nb_here = Path("multilayer_perceptron_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("iris_mlp_decision_regions.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/multilayer_perceptron/iris_mlp_decision_regions.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
print("2-feature test accuracy (viz model):", round(accuracy(y_test, viz.predict(X_test_2d)), 4))
plt.show()

## Step 8: Hidden width comparison (same 4D setup)

Refit `[4, H, 3]` for several **`H`** using the **same** hyperparameters otherwise. Wider hidden layers can help but may overfit on tiny datasets.

In [ ]:
hidden_sizes = [8, 16, 32]
rows = []

for H in hidden_sizes:
    model = MultilayerPerceptron(
        layer_sizes=[4, H, 3],
        activation="tanh",
        l2_penalty=0.01,
        rng_seed=RNG,
    )
    model.fit(X_train_s, y_train, learning_rate=LEARNING_RATE, epochs=EPOCHS)
    rows.append(
        {
            "hidden_units": H,
            "train_acc": accuracy(y_train, model.predict(X_train_s)),
            "test_acc": accuracy(y_test, model.predict(X_test_s)),
        }
    )

display(pd.DataFrame(rows).round(4))